# Need to Do

In [26]:
import geopandas as gpd
from gerrychain import Graph, Partition, updaters
from gerrychain.updaters import Tally, cut_edges
from gerrychain.proposals import recom
from gerrychain.constraints import within_percent_of_ideal_population
from gerrychain.optimization import SingleMetricOptimizer
from functools import partial
import numpy as np

In [5]:
pa_precincts = gpd.read_file("../data/cleaned/pa_cleaned_precincts") 
pa_graph = Graph.from_geodataframe(pa_precincts)

In [28]:
my_updaters = {"population": updaters.Tally("TOTPOP", alias = "population"),
                   "VAP": updaters.Tally("VAP"),
                   "Any Percent Black VAP": updaters.Tally("APBVAP", alias = "Any Percent Black VAP")
              }

initial_partition = Partition(pa_graph, assignment = "SENDIST", updaters = my_updaters)

In [2]:
# gingles = Gingleator(initial_partition, pop_col = "TOTPOP",
#                      minority_perc_col = "BVAP")

# BURST_LEN = 5
# NUM_BURSTS = 100
# # testing to make sure it works with small size
# most_VRA, observed_scores = gingles.short_burst_run(
#     num_bursts = NUM_BURSTS, num_steps = BURST_LEN, verbose = True)
# np.save(f"results/PA_50dists_BVAP_sbl{BURST_LEN}.npy", observed_scores)

# # AI credit: I asked Claude to help me read gingleator.py
# # to help me understand what the functions and args do

In [29]:
POP_TOLERANCE = 0.05  # 5% — enacted plan deviates up to ~4.3% from ideal
NUM_STEPS = 50000
ideal_pop = sum(initial_partition["population"].values()) / len(initial_partition)

proposal = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_pop,
    epsilon=POP_TOLERANCE,
    node_repeats=2,
)
chain_constraints = [within_percent_of_ideal_population(initial_partition, POP_TOLERANCE)]

In [30]:
def num_majority_black_districts(partition):
    districts = partition["population"].keys()
    majority_black_districts = 0
    for district in districts:
        district_population = partition["population"].get(district)
        black_population = partition["Any Percent Black VAP"].get(district)
        if(2 * black_population > district_population):
            majority_black_districts += 1
    return majority_black_districts

optimizer = SingleMetricOptimizer(
    proposal=proposal,
    constraints=chain_constraints,
    initial_state=initial_partition,
    optimization_metric=num_majority_black_districts,
    maximize=False
)

In [31]:
total_steps = 500
print("starting...")

# Short Bursts
min_scores_sb = np.zeros(total_steps)
scores_sb = np.zeros(total_steps)
for i, part in enumerate(optimizer.short_bursts(5, 2000, with_progress_bar=True)):
    min_scores_sb[i] = optimizer.best_score
    scores_sb[i] = optimizer.score(part)

print("done")

starting...


  0%|          | 0/10000 [00:00<?, ?it/s]


KeyError: 'APBVAP'